# 🧪 Fase 2: Análisis Exploratorio Detallado (Extenso)

Este notebook profundiza en el análisis del dataset **Diabetes 130-US Hospitals**. 
Objetivos:
1.  **Fidelidad de Datos**: Identificar inconsistencias y valores fuera de rango.
2.  **Relación con el Target**: Analizar qué variables influyen más en la readmisión.
3.  **Preparación para Modelos Generativos**: Evaluar la cardinalidad y distribución de variables para CTGAN/TVAE.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')
sns.set_style("whitegrid")
sns.set_palette("viridis")

%matplotlib inline

## 1. Carga y Primera Inspección

In [ ]:
df = pd.read_csv("../data/diabetic_data.csv")
df_nan = df.replace('?', np.nan)

print(f"Filas: {df.shape[0]}, Columnas: {df.shape[1]}")
df_nan.info()

## 2. Análisis de Datos Faltantes (Profundo)
Analizamos la redundancia de los nulos y definimos estrategias.

In [ ]:
missing = df_nan.isnull().mean() * 100
missing = missing[missing > 0].sort_values(ascending=False)

plt.figure(figsize=(12, 5))
missing.plot(kind='bar', color='skyblue')
plt.title("Porcentaje de Nulos por Variable")
plt.ylabel("% Percentage")
plt.show()

print("Variables con más del 40% de nulos (Candidatas a eliminación):")
print(missing[missing > 40])

## 3. Análisis Univariante: Variables Numéricas
Evaluamos sesgo y outliers.

In [ ]:
num_cols = ['time_in_hospital', 'num_lab_procedures', 'num_procedures', 
            'num_medications', 'number_outpatient', 'number_emergency', 
            'number_inpatient', 'number_diagnoses']

df_nan[num_cols].hist(figsize=(15, 10), bins=30, layout=(3, 3))
plt.tight_layout()
plt.show()

## 4. Análisis Bivariante: Relación con `readmitted`

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# Readmitted vs Age
sns.countplot(data=df_nan, x='age', hue='readmitted', ax=axes[0,0])
axes[0,0].set_title("Readmissions by Age Group")

# Readmitted vs Ethnicity
sns.countplot(data=df_nan, x='race', hue='readmitted', ax=axes[0,1])
axes[0,1].set_title("Readmissions by Race")

# Readmitted vs Num Inpatient (Variable Crítica)
sns.boxplot(data=df_nan, x='readmitted', y='number_inpatient', ax=axes[1,0])
axes[1,0].set_title("Distribution of Previous Inpatient Visits by Target")
axes[1,0].set_ylim(0, 10) # Zoom para evitar outliers extremos

# Readmitted vs Time in Hospital
sns.violinplot(data=df_nan, x='readmitted', y='time_in_hospital', ax=axes[1,1])
axes[1,1].set_title("Time in Hospital vs Readmission")

plt.show()

## 5. Correlaciones y Comulticolinealidad

In [ ]:
plt.figure(figsize=(10, 8))
sns.heatmap(df_nan[num_cols].corr(), annot=True, cmap='coolwarm', fmt=".2f")
plt.title("Matriz de Correlación de Variables Numéricas")
plt.show()

## 6. Calidad Médica: Altas que implican Muerte
En un estudio de readmisión, pacientes dados de alta por fallecimiento o traslado a cuidados paliativos (hospice) no deben incluirse.

In [ ]:
# Referencia: discharge_disposition_id 11, 13, 14, 19, 20, 21 suelen indicar fallecimiento/hospice
death_ids = [11, 13, 14, 19, 20, 21]
death_count = df_nan[df_nan['discharge_disposition_id'].isin(death_ids)].shape[0]
print(f"Registros de pacientes fallecidos/hospice (falsas no-readmisiones): {death_count} ({death_count/df_nan.shape[0]*100:.2f}%)")

## 7. Análisis de Pacientes Únicos vs Encuentros
Varios registros pueden pertenecer al mismo paciente (`patient_nbr`).

In [ ]:
patients_counts = df_nan['patient_nbr'].value_counts()
print(f"Pacientes totales: {len(patients_counts)}")
print(f"Pacientes con múltiples ingresos: {(patients_counts > 1).sum()}")

plt.figure(figsize=(8, 4))
sns.histplot(patients_counts, bins=30, kde=True)
plt.title("Número de ingresos por paciente")
plt.xlabel("Ingresos")
plt.yscale('log')
plt.show()

## 8. Medicamentos de Diabetes: Análisis de Tendencia
Analizamos los 24 medicamentos.

In [ ]:
meds = ['metformin', 'repaglinide', 'nateglinide', 'chlorpropamide', 'glimepiride', 
        'acetohexamide', 'glipizide', 'glyburide', 'tolbutamide', 'pioglitazone', 
        'rosiglitazone', 'acarbose', 'miglitol', 'troglitazone', 'tolazamide', 
        'examide', 'citoglipton', 'insulin', 'glyburide-metformin', 'glipizide-metformin', 
        'glimepiride-pioglitazone', 'metformin-rosiglitazone', 'metformin-pioglitazone']

med_counts = {}
for m in meds:
    med_counts[m] = (df_nan[m] != 'No').sum()

med_df = pd.Series(med_counts).sort_values(ascending=False)
plt.figure(figsize=(12, 6))
med_df.plot(kind='bar')
plt.title("Uso de Medicamentos (Cualquier dosis)")
plt.yscale('log')
plt.ylabel("Count (Log Scale)")
plt.show()